## Suggestions
1. Only dense layers
2. On a real dataset

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. Deterministic Multi-Bit Quantization (No Stochastic)
# ---------------------------------------------------------
def quantize_tensor_pt(w, bits):
    """Deterministic uniform quantization (round to nearest)."""
    w_min, w_max = w.min(), w.max()
    if w_max == w_min:
        return w

    scale = (w_max - w_min) / (2**bits - 1)
    scaled = (w - w_min) / scale

    q = torch.round(scaled)
    q = torch.clamp(q, 0, 2**bits - 1)

    w_q = (q * scale) + w_min
    # Straight-Through Estimator (gradient passes through)
    return w + (w_q - w).detach()

def quantize_pt(w, bits=4, per_channel=True):
    """Apply quantization to weight tensor with optional per‑channel scaling."""
    if w.ndim == 1 or not per_channel:
        return quantize_tensor_pt(w, bits)

    w_q = torch.zeros_like(w)
    for i in range(w.shape[0]):  # per‑output‑channel
        w_q[i] = quantize_tensor_pt(w[i], bits)
    return w_q

def apply_custom_quantization(model, bits=4, per_channel=True):
    """Return a copy of the model with weights quantized to `bits` (deterministic)."""
    q_model = copy.deepcopy(model).eval()
    with torch.no_grad():
        for name, module in q_model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.weight.data = quantize_pt(
                    module.weight.data,
                    bits=bits,
                    per_channel=per_channel
                )
    return q_model

# ---------------------------------------------------------
# 2. CNN Architecture
# ---------------------------------------------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 28 * 28, 64)
        self.relu4 = nn.ReLU()
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.flatten(x)
        x = self.relu4(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------------------------------------------------------
# 3. Training
# ---------------------------------------------------------
def train_model(model):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(3):
        for _ in range(20):
            x = torch.randn(16, 3, 224, 224)
            y = torch.randint(0, 10, (16,))
            out = model(x)
            loss = loss_fn(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    return model

# ---------------------------------------------------------
# 4. Math Utilities
# ---------------------------------------------------------
def calculate_condition_number(w):
    w = w.detach().float()
    if w.dim() > 2:
        w = w.view(w.size(0), -1)
    s = torch.linalg.svdvals(w)
    return (s[0] / (s[-1] + 1e-8)).item()

def relative_error(x, y):
    return (torch.norm(x - y) / (torch.norm(x) + 1e-8)).item()

# ---------------------------------------------------------
# 5. Layer-wise Activation Extraction
# ---------------------------------------------------------
def get_activations(model, x):
    """Return a list of intermediate activations for each trainable layer."""
    activations = []
    with torch.no_grad():
        x = model.conv1(x)
        x = model.relu1(x)
        x = model.pool1(x)
        activations.append(x)

        x = model.conv2(x)
        x = model.relu2(x)
        x = model.pool2(x)
        activations.append(x) 

        x = model.conv3(x)
        x = model.relu3(x)
        x = model.pool3(x)
        activations.append(x) 

        x = model.flatten(x)
        x = model.fc1(x)
        x = model.relu4(x)
        activations.append(x)  

        x = model.fc2(x)
        activations.append(x)

    return activations

# ---------------------------------------------------------
# 6. MAIN PIPELINE
# ---------------------------------------------------------
def run_pipeline():
    model_fp32 = train_model(SimpleCNN())

    bit_widths = [8, 4, 2]
    test_input = torch.randn(1, 3, 224, 224)

    fp32_activations = get_activations(model_fp32, test_input)
    fp32_final = fp32_activations[-1]

    layer_names = ["conv1", "conv2", "conv3", "fc1", "fc2"]

    summary = []

    for bits in bit_widths:
        print(f"\n{'#'*80}")
        print(f"###  {bits}-BIT QUANTIZATION (Deterministic Rounding)")
        print(f"{'#'*80}")

        # Quantize model
        model_q = apply_custom_quantization(model_fp32, bits=bits)
        q_activations = get_activations(model_q, test_input)

        print("\n📋 Layer‑by‑Layer Metrics:")
        print("-" * 80)
        print(f"{'Layer':<10} {'Condition κ':>14} {'Weight Error':>14} {'Activation Drift':>18}")
        print("-" * 80)

        layer_kappas = []
        layer_weight_errs = []
        layer_drifts = []

        # Get weight modules for condition number / weight error
        fp_modules = {name: mod for name, mod in model_fp32.named_modules()
                      if isinstance(mod, (nn.Conv2d, nn.Linear))}
        q_modules = {name: mod for name, mod in model_q.named_modules()
                     if isinstance(mod, (nn.Conv2d, nn.Linear))}

        for idx, (name, fp_mod) in enumerate(fp_modules.items()):
            q_mod = q_modules[name]
            w_fp = fp_mod.weight.detach()
            w_q = q_mod.weight.detach()

            kappa = calculate_condition_number(w_fp)
            w_err = relative_error(w_fp, w_q)

            # Activation drift for the corresponding layer output
            act_drift = relative_error(fp32_activations[idx], q_activations[idx])

            layer_kappas.append(kappa)
            layer_weight_errs.append(w_err)
            layer_drifts.append(act_drift)

            # Use the layer name from our list (align order)
            display_name = layer_names[idx] if idx < len(layer_names) else name
            print(f"{display_name:<10} {kappa:>14.2f} {w_err:>14.4f} {act_drift:>18.4f}")

        # Final output drift
        final_drift = relative_error(fp32_final, q_activations[-1])

        # Sensitivity test
        noise = 0.01 * torch.randn_like(test_input)
        test_input_noisy = test_input + noise
        fp32_noisy = get_activations(model_fp32, test_input_noisy)[-1]
        q_noisy = get_activations(model_q, test_input_noisy)[-1]
        fp_sens = relative_error(fp32_final, fp32_noisy)
        q_sens = relative_error(q_activations[-1], q_noisy)

        # Correlations
        corr_kappa_werr = np.corrcoef(layer_kappas, layer_weight_errs)[0, 1]
        corr_kappa_drift = np.corrcoef(layer_kappas, layer_drifts)[0, 1]

        print("-" * 80)
        print(f"🔹 Final Output Drift:        {final_drift:.4f}")
        print(f"🔹 FP32 Sensitivity to Noise: {fp_sens:.4f}")
        print(f"🔹 Q{bits} Sensitivity to Noise:  {q_sens:.4f}")
        print(f"📊 Correlation κ vs Weight Error:      {corr_kappa_werr:.4f}")
        print(f"📊 Correlation κ vs Activation Drift:  {corr_kappa_drift:.4f}")

        # Store summary
        summary.append({
            'bits': bits,
            'final_drift': final_drift,
            'fp_sens': fp_sens,
            'q_sens': q_sens,
            'corr_kappa_werr': corr_kappa_werr,
            'corr_kappa_drift': corr_kappa_drift,
            'avg_weight_err': np.mean(layer_weight_errs),
            'avg_act_drift': np.mean(layer_drifts),
            'layer_kappas': layer_kappas,
            'layer_weight_errs': layer_weight_errs,
            'layer_drifts': layer_drifts
        })

    # ---------------------------------------------------------
    # Final Summary Table
    # ---------------------------------------------------------
    print("\n\n" + "=" * 80)
    print("📊 FINAL SUMMARY – Deterministic Quantization at 8, 4, 2 bits")
    print("=" * 80)
    print(f"{'Bits':<6} {'Final Drift':>12} {'FP Sens':>10} {'Q Sens':>10} "
          f"{'κ-Werr Corr':>14} {'κ-Drift Corr':>16} {'Avg W Err':>12} {'Avg Act Drift':>15}")
    print("-" * 80)
    for s in summary:
        print(f"{s['bits']:<6} {s['final_drift']:>12.4f} {s['fp_sens']:>10.4f} {s['q_sens']:>10.4f} "
              f"{s['corr_kappa_werr']:>14.4f} {s['corr_kappa_drift']:>16.4f} "
              f"{s['avg_weight_err']:>12.4f} {s['avg_act_drift']:>15.4f}")

    # Optional: Print a table of κ values for reference
    print("\n" + "=" * 80)
    print("📌 Layer Condition Numbers (κ) — same for all bit‑widths")
    print("=" * 80)
    for name, kappa in zip(layer_names, summary[0]['layer_kappas']):
        print(f"{name:<10} κ = {kappa:.2f}")

if __name__ == "__main__":
    run_pipeline()


################################################################################
###  8-BIT QUANTIZATION (Deterministic Rounding)
################################################################################

📋 Layer‑by‑Layer Metrics:
--------------------------------------------------------------------------------
Layer         Condition κ   Weight Error   Activation Drift
--------------------------------------------------------------------------------
conv1                4.75         0.0034             0.0027
conv2                2.73         0.0038             0.0023
conv3                3.64         0.0039             0.0318
fc1                  9.47         0.0060             0.0087
fc2                  1.87         0.0040             0.0003
--------------------------------------------------------------------------------
🔹 Final Output Drift:        0.0003
🔹 FP32 Sensitivity to Noise: 0.0001
🔹 Q8 Sensitivity to Noise:  0.0001
📊 Correlation κ vs Weight Error:      0.8479
📊 Corr